# Document and Media Base Types Reference

Developer-facing statements defined in `langchain_core.documents.base`.

# `PathLike`

Path value accepted by blob-loading APIs.

```python
PathLike = str | PurePath
```

---

# `BaseMedia: Serializable`

Base model for content used in retrieval and data-processing workflows.

## Fields

```python
id: str | None = Field(default=None, coerce_numbers_to_str=True) # Optional content identifier; numeric values are coerced to strings
metadata: dict[Any, Any] = Field(default_factory=dict) # Arbitrary metadata associated with the content
```

The identifier is not required or validated as globally unique.

---

# `Blob: BaseMedia`

Immutable raw-data abstraction used by document loaders and parsers.

A blob can hold in-memory text or bytes, or refer to a file through `path`. Arbitrary field types are allowed and the model is frozen after creation.

## Fields

```python
data: bytes | str | None = None # In-memory raw data
mimetype: str | None = None # MIME type of the data
encoding: str = "utf-8" # Encoding used when converting between bytes and text
path: PathLike | None = None # Original file or content location
```

Also inherits `id` and `metadata` from `BaseMedia`.

At validation time, at least one of `data` or `path` must be supplied. Otherwise, construction raises `ValueError`.

## Properties

### `source`

Returns the blob source as `str | None`.

A `"source"` value in `metadata` takes precedence over `path`; otherwise, the path is converted to a string.

## Methods

### `as_string`

Reads the blob as text.

```python
as_string(
    self,
) -> str # Blob content as a string
```

File-backed blobs are read using `encoding`. Byte data is decoded, while string data is returned unchanged.

Raises `ValueError` when the blob cannot be represented as a string.

### `as_bytes`

Reads the blob as bytes.

```python
as_bytes(
    self,
) -> bytes # Blob content as bytes
```

String data is encoded using `encoding`, while file-backed blobs are read as bytes.

Raises `ValueError` when the blob cannot be represented as bytes.

### `as_bytes_io`

Provides the blob as a byte stream context manager.

```python
as_bytes_io(
    self,
) -> Generator[BytesIO | BufferedReader, None, None] # Generator yielding an in-memory or file-backed byte stream
```

Byte data produces a `BytesIO` stream. File-backed data yields a binary file reader.

Raises `NotImplementedError` for unsupported data, including in-memory string content.

### `from_path`

Creates a file-backed blob without immediately reading the file.

```python
@classmethod
from_path(
    cls,
    path: PathLike, # File represented by the blob
    *,
    encoding: str = "utf-8", # Encoding used when the file is later read as text
    mime_type: str | None = None, # Explicit MIME type
    guess_type: bool = True, # Guess the MIME type from the file extension when none is supplied
    metadata: dict[Any, Any] | None = None, # Metadata associated with the blob
) -> Blob # File-backed blob
```

### `from_data`

Creates a blob from in-memory text or bytes.

```python
@classmethod
from_data(
    cls,
    data: str | bytes, # In-memory blob content
    *,
    encoding: str = "utf-8", # Encoding used for text and byte conversion
    mime_type: str | None = None, # MIME type of the data
    path: str | None = None, # Optional source location associated with the data
    metadata: dict[Any, Any] | None = None, # Metadata associated with the blob
) -> Blob # In-memory blob
```

## Behaviour

The blob representation includes its object identity and appends its resolved source when available.

---

# `Document: BaseMedia`

Stores a piece of text and its associated retrieval metadata.

This type is intended for retrieval workflows rather than LLM chat messages.

## Fields

```python
page_content: str # Document text
type: Literal["Document"] = "Document" # Serializable document discriminator
```

Also inherits `id` and `metadata` from `BaseMedia`.

## Constructor

```python
Document(
    page_content: str, # Document text
    **kwargs: Any, # Additional model fields such as id and metadata
) -> None
```

`page_content` may be passed positionally or by name.

## Methods

### `is_lc_serializable`

Returns `True`, marking the class as LangChain serializable.

### `get_lc_namespace`

Returns the serialization namespace `["langchain", "schema", "document"]`.

## Behaviour

Its string representation contains `page_content` and includes `metadata` only when metadata is non-empty. The inherited `id` field is intentionally omitted from this representation.

In [ ]:
#%pip install -U langchain-core#Run this once if LangChain Core is not installed

from pathlib import Path#Import Path for creating and managing the sample text file
from langchain_core.documents import Blob, Document#Import Blob and Document from LangChain Core

file_path = Path("company_policy.txt")#Define the path of the sample text file

sample_text = """Employees may work remotely for up to three days per week.

All leave requests must be submitted at least seven days in advance.

Company devices must not be shared with unauthorized individuals."""#Create sample company-policy content

file_path.write_text(sample_text, encoding="utf-8")#Save the sample content inside the text file

blob = Blob.from_path(#Create a file-backed Blob without immediately reading the file
    path=file_path,#Provide the path of the source file
    encoding="utf-8",#Specify the text-file encoding
    mime_type="text/plain",#Specify the MIME type of the file
    metadata={"department": "Human Resources"},#Attach additional source metadata
)

text = blob.as_string()#Read the Blob content as a string
paragraphs = text.split("\n\n")#Split the text whenever an empty line appears
documents = []#Create an empty list for the generated Documents

for paragraph_number, paragraph in enumerate(paragraphs, start=1):#Process every paragraph with its position
    cleaned_paragraph = paragraph.strip()#Remove unnecessary spaces and newline characters

    if cleaned_paragraph:#Continue only when the paragraph contains text
        document = Document(#Create a LangChain Document for the paragraph
            page_content=cleaned_paragraph,#Store the paragraph as the document content
            metadata={#Create metadata describing the document
                "source": blob.source,#Store the original file path
                "mime_type": blob.mimetype,#Store the MIME type
                "department": blob.metadata["department"],#Copy the department metadata from the Blob
                "paragraph_number": paragraph_number,#Store the paragraph position
            },#Finish the metadata dictionary
        )#Finish creating the Document

        documents.append(document)#Add the Document to the result list

print(f"Blob source: {blob.source}")#Display the resolved Blob source
print(f"Blob MIME type: {blob.mimetype}")#Display the Blob MIME type
print(f"Total documents: {len(documents)}")#Display the number of generated Documents
print("=" * 60)#Display a separator line

for document_number, document in enumerate(documents, start=1):#Process every generated Document
    print(f"Document {document_number}")#Display the current document number
    print(f"Content: {document.page_content}")#Display the document text
    print(f"Metadata: {document.metadata}")#Display the document metadata
    print("-" * 60)#Display a separator after the document